# Added Portfolio Section: SQL + Sales Analysis + Excel Data Analysis

This section extends the original Decision Tree / Heart Disease notebook with a separate, business-focused Sales Analysis workflow. It is suitable for demonstrating basic SQL, Python/Pandas, Excel analysis, visualisation, and business recommendations.

**Important:** The sales records below are synthetic sample data created for learning. They are not real customer transactions.

## 1. Create a sample sales dataset

This creates a reproducible dataset so the added sections can run without needing a separate sales CSV download.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
from pathlib import Path

sales_data = {
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
                1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020],
    "OrderDate": ["2025-01-05", "2025-01-12", "2025-01-18", "2025-02-03", "2025-02-10",
                  "2025-02-17", "2025-03-02", "2025-03-11", "2025-03-19", "2025-04-04",
                  "2025-04-13", "2025-04-22", "2025-05-06", "2025-05-14", "2025-05-27",
                  "2025-06-03", "2025-06-12", "2025-06-21", "2025-06-25", "2025-06-29"],
    "Region": ["West", "North", "South", "West", "East", "North", "South", "West", "East", "North",
               "West", "South", "East", "North", "West", "South", "East", "West", "North", "South"],
    "Product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Laptop", "Mouse", "Monitor", "Keyboard",
                "Laptop", "Mouse", "Monitor", "Laptop", "Keyboard", "Laptop", "Mouse", "Monitor",
                "Keyboard", "Laptop", "Monitor", "Mouse"],
    "Category": ["Electronics", "Accessories", "Accessories", "Electronics", "Electronics", "Accessories",
                 "Electronics", "Accessories", "Electronics", "Accessories", "Electronics", "Electronics",
                 "Accessories", "Electronics", "Accessories", "Electronics", "Accessories", "Electronics",
                 "Electronics", "Accessories"],
    "Quantity": [2, 5, 3, 1, 1, 8, 2, 4, 1, 6, 2, 1, 5, 2, 10, 1, 7, 1, 2, 4],
    "UnitPrice": [55000, 700, 1200, 14000, 58000, 650, 13500, 1100, 56000, 750,
                  14500, 54000, 1000, 59000, 680, 15000, 1250, 57000, 14800, 720]
}

sales_df = pd.DataFrame(sales_data)
sales_df["OrderDate"] = pd.to_datetime(sales_df["OrderDate"])
sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]
sales_df.head()

In [ ]:
# Quick dataset checks
print("Rows and columns:", sales_df.shape)
display(sales_df.info())
display(sales_df.isnull().sum())
display(sales_df.duplicated().sum())
display(sales_df.describe(include="all"))

## 2. Sales analysis using Pandas

Explore overall performance, monthly revenue, product performance, and regional performance.

In [ ]:
# Key Performance Indicators (KPIs)
total_revenue = sales_df["Revenue"].sum()
total_orders = sales_df["OrderID"].nunique()
total_units = sales_df["Quantity"].sum()
average_order_value = sales_df.groupby("OrderID")["Revenue"].sum().mean()

print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Units Sold: {total_units}")
print(f"Average Order Value: ₹{average_order_value:,.2f}")

In [ ]:
# Revenue by product
product_summary = (
    sales_df.groupby("Product", as_index=False)
    .agg(Total_Quantity=("Quantity", "sum"), Total_Revenue=("Revenue", "sum"))
    .sort_values("Total_Revenue", ascending=False)
)
display(product_summary)

In [ ]:
# Revenue by region
region_summary = (
    sales_df.groupby("Region", as_index=False)
    .agg(Orders=("OrderID", "nunique"), Revenue=("Revenue", "sum"))
    .sort_values("Revenue", ascending=False)
)
display(region_summary)

In [ ]:
# Monthly revenue trend
sales_df["Month"] = sales_df["OrderDate"].dt.to_period("M").astype(str)
monthly_summary = (
    sales_df.groupby("Month", as_index=False)["Revenue"]
    .sum()
    .sort_values("Month")
)
display(monthly_summary)

plt.figure(figsize=(9, 4))
plt.plot(monthly_summary["Month"], monthly_summary["Revenue"], marker="o")
plt.title("Monthly Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visual comparison: revenue by product
plt.figure(figsize=(8, 4))
plt.bar(product_summary["Product"], product_summary["Total_Revenue"])
plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 3. SQL analysis using SQLite

The notebook creates an in-memory SQLite database and loads the sales table into it. These are practical SQL examples using `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, and aggregate functions. The same concepts transfer to MS SQL Server with small syntax changes if needed.

In [ ]:
# Load the sales dataset into an in-memory SQLite database
connection = sqlite3.connect(":memory:")
sales_sql_df = sales_df.copy()
sales_sql_df["OrderDate"] = sales_sql_df["OrderDate"].dt.strftime("%Y-%m-%d")
sales_sql_df.to_sql("sales", connection, if_exists="replace", index=False)

# 1. View records
display(pd.read_sql_query("SELECT * FROM sales LIMIT 5;", connection))

# 2. Overall sales KPIs
display(pd.read_sql_query(
    """SELECT COUNT(DISTINCT OrderID) AS total_orders,
              SUM(Quantity) AS units_sold,
              ROUND(SUM(Revenue), 2) AS total_revenue,
              ROUND(AVG(Revenue), 2) AS average_line_revenue
       FROM sales;""",
    connection
))

# 3. Revenue by product
display(pd.read_sql_query(
    """SELECT Product,
              SUM(Quantity) AS units_sold,
              SUM(Revenue) AS total_revenue
       FROM sales
       GROUP BY Product
       ORDER BY total_revenue DESC;""",
    connection
))

# 4. Revenue by region
display(pd.read_sql_query(
    """SELECT Region, SUM(Revenue) AS total_revenue
       FROM sales
       GROUP BY Region
       ORDER BY total_revenue DESC;""",
    connection
))

# 5. Filter high-value order lines
display(pd.read_sql_query(
    "SELECT OrderID, OrderDate, Product, Revenue FROM sales WHERE Revenue >= 20000 ORDER BY Revenue DESC;",
    connection
))

In [ ]:
# Optional SQL practice: monthly sales
monthly_sql = pd.read_sql_query(
    """SELECT SUBSTR(OrderDate, 1, 7) AS month,
              SUM(Revenue) AS monthly_revenue
       FROM sales
       GROUP BY SUBSTR(OrderDate, 1, 7)
       ORDER BY month;""",
    connection
)
display(monthly_sql)

## 4. Excel data analysis

This section exports the sales data to an Excel workbook, then reads it back and performs common Excel-style analysis in Python. If Excel files cannot be read or written, install the required engine with `pip install openpyxl`.

You can also open the generated workbook in Microsoft Excel and practise filters, sorting, PivotTables, conditional formatting, and charts.

In [ ]:
# Export raw sales data and summary sheets to an Excel workbook
excel_path = "sample_sales_analysis.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    sales_df.to_excel(writer, sheet_name="Sales_Data", index=False)
    product_summary.to_excel(writer, sheet_name="Product_Summary", index=False)
    region_summary.to_excel(writer, sheet_name="Region_Summary", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly_Summary", index=False)

print(f"Excel workbook saved to: {Path(excel_path).resolve()}")

In [ ]:
# Read the Excel data back into pandas
excel_sales_df = pd.read_excel(excel_path, sheet_name="Sales_Data")
excel_sales_df["OrderDate"] = pd.to_datetime(excel_sales_df["OrderDate"])

print("Excel data shape:", excel_sales_df.shape)
display(excel_sales_df.head())
display(excel_sales_df.isnull().sum())

In [ ]:
# Excel-style analysis: sort, filter, and summary
top_revenue_rows = excel_sales_df.sort_values("Revenue", ascending=False).head(5)
high_value_rows = excel_sales_df[excel_sales_df["Revenue"] >= 20000]

print("Top 5 sales lines by revenue")
display(top_revenue_rows[["OrderID", "OrderDate", "Region", "Product", "Revenue"]])

print("Sales lines with revenue >= ₹20,000")
display(high_value_rows[["OrderID", "Region", "Product", "Quantity", "Revenue"]])

# Pivot-table-style summary (similar to an Excel PivotTable)
excel_pivot = pd.pivot_table(
    excel_sales_df,
    values="Revenue",
    index="Region",
    columns="Category",
    aggfunc="sum",
    fill_value=0,
    margins=True
)
display(excel_pivot)

## 5. Business insights and recommendations

Use the outputs above to write evidence-based observations. Do not copy generic statements; fill in the actual product/region names and values produced by your run.

Suggested structure:
1. **Overall performance:** report total revenue, number of orders, and units sold.
2. **Top product:** identify the product with the highest total revenue.
3. **Regional performance:** compare regions and identify the strongest region.
4. **Monthly trend:** identify the strongest and weakest months in this sample.
5. **Recommendation:** suggest one action supported by the results, such as reviewing stock levels for high-revenue products or investigating a low-performing region.

Because this is synthetic sample data, do not present the results as real company performance.

In [ ]:
# Generate data-backed insight starters
top_product = product_summary.iloc[0]
top_region = region_summary.iloc[0]
best_month = monthly_summary.loc[monthly_summary["Revenue"].idxmax()]

print(f"Top product by revenue: {top_product['Product']} (₹{top_product['Total_Revenue']:,.2f})")
print(f"Top region by revenue: {top_region['Region']} (₹{top_region['Revenue']:,.2f})")
print(f"Highest-revenue month: {best_month['Month']} (₹{best_month['Revenue']:,.2f})")
print("\nWrite your own business recommendation based on these findings.")

## 6. Resume / GitHub project description

**Project title:** Sales Performance Analysis using SQL, Python, and Excel

**Tools:** Python, Pandas, SQLite SQL, Excel, Matplotlib, Jupyter Notebook.

**Resume bullet template (edit after running the notebook):**
- Analysed a sales dataset using SQL, Pandas, and Excel to calculate revenue KPIs and compare product, regional, and monthly performance.
- Built summary tables and visualisations to communicate sales trends and generate data-backed business recommendations.
- Automated Excel workbook creation with separate raw-data and summary sheets.

Only include metrics or outcomes you have personally verified. Clearly label this as a practice project using synthetic data unless you replace it with a real, properly sourced dataset.

In [ ]:
# Close the SQLite connection when you have finished SQL practice
connection.close()